# Modularização de programas — Tutorial

**Programação C (COMP0512) — UFS — 2026.2**

Este tutorial pega o `notas.c` da aula passada — aquele que calcula as médias de uma turma
cujo tamanho só se conhece rodando — e o quebra em módulos que compilam separado. Depois de
montar a versão modular, você vai ver o que sobra dentro de um `.o`, provocar de propósito os
dois erros de ligação, automatizar tudo com um `Makefile` e empacotar o resultado como
biblioteca.

## Objetivos

Ao final deste tutorial você será capaz de:

- Separar um programa em `.h` (interface) e `.c` (implementação), com guardas de inclusão;
- Compilar em duas etapas (`-c` e ligação) e explicar o que cada uma produz;
- Ler a lista de símbolos de um `.o` e diagnosticar `undefined reference` e `multiple definition`;
- Escrever um `Makefile` que recompila só o que mudou;
- Gerar e usar uma biblioteca estática (`.a`) e uma compartilhada (`.so`) com `-I`, `-L` e `-l`.

> Os comandos abaixo supõem Linux (é o caso do Colab). No macOS, `.so` vira `.dylib`,
> `ldd` vira `otool -L` e `LD_LIBRARY_PATH` vira `DYLD_LIBRARY_PATH`.


## Preparando o ambiente

Tudo acontece dentro do diretório `modular/`. Rode esta célula uma vez.


In [ ]:
import os
os.makedirs('modular', exist_ok=True)
%cd modular
!gcc --version | head -1
!make --version | head -1


## 1. O programa da aula, agora em quatro arquivos

O `notas.c` fazia três coisas: cuidava da matriz, calculava estatísticas e imprimia o
relatório. Cada uma vira um módulo com o seu par `.h`/`.c`, e o `main.c` fica só com a
montagem.

Repare em três detalhes enquanto escreve os arquivos:

1. todo header começa com `#ifndef`/`#define` e termina com `#endif`;
2. todo `.c` inclui o **próprio** header — é assim que o compilador confere o contrato;
3. `main.c` inclui só `turma.h`: ele não precisa saber que `matriz` e `estat` existem.


In [ ]:
%%writefile matriz.h
/* matriz.h --- interface: alocar e liberar matrizes de double */
#ifndef MATRIZ_H
#define MATRIZ_H

double **cria_matriz(int nl, int nc);
void libera_matriz(double **m, int nl);

#endif /* MATRIZ_H */


In [ ]:
%%writefile matriz.c
/* matriz.c --- implementacao das matrizes dinamicas */
#include <stdlib.h>
#include "matriz.h"

double **cria_matriz(int nl, int nc)
{
    double **m = malloc(nl * sizeof(double *));
    if (m == NULL)
        return NULL;
    for (int i = 0; i < nl; i++) {
        m[i] = malloc(nc * sizeof(double));
        if (m[i] == NULL) {             /* falhou no meio do caminho */
            for (int k = 0; k < i; k++)
                free(m[k]);
            free(m);
            return NULL;
        }
    }
    return m;
}

void libera_matriz(double **m, int nl)
{
    for (int i = 0; i < nl; i++)
        free(m[i]);
    free(m);
}


In [ ]:
%%writefile estat.h
/* estat.h --- interface: estatisticas de um vetor de notas */
#ifndef ESTAT_H
#define ESTAT_H

double media(const double *v, int n);
double desvio_padrao(const double *v, int n);

#endif /* ESTAT_H */


In [ ]:
%%writefile estat.c
/* estat.c --- implementacao das estatisticas */
#include <math.h>          /* sqrt --- vai cobrar o -lm na ligacao */
#include "estat.h"

double media(const double *v, int n)
{
    double s = 0.0;
    for (int i = 0; i < n; i++)
        s += v[i];
    return n > 0 ? s / n : 0.0;
}

double desvio_padrao(const double *v, int n)
{
    double mu = media(v, n), s = 0.0;
    for (int i = 0; i < n; i++)
        s += (v[i] - mu) * (v[i] - mu);
    return n > 0 ? sqrt(s / n) : 0.0;
}


In [ ]:
%%writefile turma.h
/* turma.h --- interface: a turma e seu relatorio */
#ifndef TURMA_H
#define TURMA_H

typedef struct {
    double **notas;
    int nalunos;
    int navaliacoes;
} Turma;

Turma *turma_cria(int nalunos, int navaliacoes);
void turma_libera(Turma *t);
void turma_relatorio(const Turma *t);

#endif /* TURMA_H */


In [ ]:
%%writefile turma.c
/* turma.c --- implementacao da turma: usa matriz.h e estat.h */
#include <stdio.h>
#include <stdlib.h>
#include "turma.h"
#include "matriz.h"
#include "estat.h"

static void cabecalho(void)          /* so turma.c enxerga */
{
    printf("%-8s %8s %8s\n", "aluno", "media", "desvio");
}

Turma *turma_cria(int nalunos, int navaliacoes)
{
    Turma *t = malloc(sizeof(Turma));
    if (t == NULL)
        return NULL;
    t->notas = cria_matriz(nalunos, navaliacoes);
    if (t->notas == NULL) {
        free(t);
        return NULL;
    }
    t->nalunos = nalunos;
    t->navaliacoes = navaliacoes;
    return t;
}

void turma_libera(Turma *t)
{
    if (t == NULL)
        return;
    libera_matriz(t->notas, t->nalunos);
    free(t);
}

void turma_relatorio(const Turma *t)
{
    cabecalho();
    for (int i = 0; i < t->nalunos; i++)
        printf("%-8d %8.2f %8.2f\n", i,
               media(t->notas[i], t->navaliacoes),
               desvio_padrao(t->notas[i], t->navaliacoes));
}


In [ ]:
%%writefile main.c
/* main.c --- monta a turma e pede o relatorio */
#include <stdio.h>
#include "turma.h"        /* nao inclui matriz.h nem estat.h */

int main(void)
{
    double valores[3][4] = {
        { 8.0, 6.5, 9.0, 7.0 },
        { 5.0, 7.0, 4.0, 7.0 },
        { 9.0, 10.0, 8.5, 9.5 }
    };
    Turma *t = turma_cria(3, 4);
    if (t == NULL) {
        fprintf(stderr, "sem memoria\n");
        return 1;
    }
    for (int i = 0; i < t->nalunos; i++)
        for (int j = 0; j < t->navaliacoes; j++)
            t->notas[i][j] = valores[i][j];

    turma_relatorio(t);
    turma_libera(t);
    return 0;
}


### Compilar em duas etapas

Primeiro `-c` transforma cada `.c` em um `.o`; depois a ligação junta os quatro em um
executável. A saída tem que bater com a da aula passada.


In [ ]:
!gcc -Wall -Wextra -c matriz.c estat.c turma.c main.c
!ls -l *.o
!gcc matriz.o estat.o turma.o main.o -o notas -lm
!./notas
!./notas | grep -q '^0 *7.62 *0.96' && echo OK \
  || echo 'Verifique: esperava media 7.62 e desvio 0.96 para o aluno 0'


## 2. O header é substituição de texto

`#include` não é mágica: o pré-processador troca a linha pelo conteúdo do arquivo. Dá para
ver isso parando a compilação logo depois dele, com `gcc -E`.


In [ ]:
%%writefile mini.c
#include "matriz.h"
int main(void) { return 0; }


In [ ]:
!gcc -E mini.c


As linhas que começam com `#` são apenas marcações de origem ("daqui em diante o texto
veio de `matriz.h`"). O resto é o seu header, copiado inteiro.

Troque `"matriz.h"` por `<stdio.h>` na célula acima e conte as linhas com
`!gcc -E mini.c | wc -l`. É esse texto gigante que o compilador recebe a cada arquivo.


### Guardas de inclusão: o que elas evitam

Um header sem guarda, incluído duas vezes, define os mesmos tipos duas vezes. Vamos provocar
o erro e depois consertá-lo.


In [ ]:
%%writefile ponto_sem_guarda.h
/* sem #ifndef de proposito */
typedef struct { double x, y; } Ponto;


In [ ]:
%%writefile usa_ponto.c
#include "ponto_sem_guarda.h"
#include "ponto_sem_guarda.h"     /* a segunda inclusao e o problema */
int main(void) { Ponto p = { 1.0, 2.0 }; return (int) p.x; }


In [ ]:
# esperado: erro de compilacao (redefinition of ...)
!gcc -Wall usa_ponto.c -o usa_ponto || echo '>>> falhou, como esperado'


> TODO: reescreva ponto_sem_guarda.h COM a guarda de inclusao e rode de novo.
> Dica: #ifndef PONTO_H / #define PONTO_H / ... / #endif


In [ ]:
%%writefile ponto_sem_guarda.h
/* TODO: acrescente a guarda aqui */
typedef struct { double x, y; } Ponto;


In [ ]:
!gcc -Wall usa_ponto.c -o usa_ponto && echo 'OK: compilou com a guarda' \
  || echo 'Ainda nao: a guarda nao esta protegendo o header'


## 3. O que sobra dentro de um `.o`

Um arquivo-objeto carrega o código de máquina e duas listas de símbolos: o que ele **oferece**
e o que ele ainda **deve**. O `nm` mostra as duas.

| letra | significado |
|---|---|
| `T` | definido aqui, visível para o ligador |
| `t` | definido aqui, mas `static` — ninguém de fora enxerga |
| `U` | usado aqui, definido em algum outro lugar |


In [ ]:
!nm turma.o | sort -k 3


Confira: `turma_cria`, `turma_libera` e `turma_relatorio` estão como `T`; `cabecalho`,
que é `static`, aparece como `t` minúsculo; `media`, `cria_matriz` e `printf` estão como `U`.

**Experimento:** apague o `static` de `cabecalho` em `turma.c`, recompile e rode o `nm` de
novo. Qual letra ela passa a ter?


In [ ]:
# Exercicio 1: descubra, sem abrir os fontes, quais simbolos main.o ainda deve
# TODO: complete o comando para listar SO os simbolos indefinidos de main.o
# Dica: nm tem uma opcao para isso (veja `nm --help`)
!nm main.o


## 4. Os dois erros de ligação

### `undefined reference`: faltou quem oferece


In [ ]:
# ligando sem turma.o: main.o fica devendo turma_cria e companhia
!gcc main.o matriz.o estat.o -o quebrado -lm || echo '>>> erro de ligacao, como esperado'


### `multiple definition`: sobrou quem oferece

O clássico: uma **definição** de variável dentro de um header, incluído por dois `.c`.


In [ ]:
%%writefile contador.h
#ifndef CONTADOR_H
#define CONTADOR_H
int contador = 0;      /* ERRADO: isto e uma definicao, nao uma declaracao */
#endif


In [ ]:
%%writefile a.c
#include "contador.h"
int main(void) { return contador; }


In [ ]:
%%writefile b.c
#include "contador.h"
void usa(void) { contador++; }


In [ ]:
!gcc -fno-common -Wall a.c b.c -o duplicado || echo '>>> multiple definition, como esperado'


> Exercicio 2: conserte sem mudar a.c nem b.c
> TODO: deixe no header apenas a DECLARACAO (extern) e crie contador.c com a definicao


In [ ]:
%%writefile contador.h
#ifndef CONTADOR_H
#define CONTADOR_H
int contador = 0;      /* TODO: trocar por extern */
#endif


> TODO: escreva contador.c com a unica definicao de contador


In [ ]:
%%writefile contador.c
/* TODO */


In [ ]:
!gcc -fno-common -Wall a.c b.c contador.c -o duplicado && echo 'OK: uma definicao so' \
  || echo 'Ainda nao: releia a diferenca entre declarar e definir'


## 5. Automatizando com `make`

O `make` compara datas: se o alvo é mais velho que algum pré-requisito, ele roda a receita.
O trabalho de quem escreve o `Makefile` é dizer **de que cada coisa depende** — em especial,
de quais headers cada objeto depende, porque isso o `make` não adivinha.

⚠️ A receita precisa começar com um TAB. A célula abaixo já usa TAB; se você reescrever,
cuidado para o editor não trocar por espaços.


In [ ]:
%%writefile Makefile
CC      = gcc
CFLAGS  = -Wall -Wextra -g
LDLIBS  = -lm
OBJ     = main.o turma.o matriz.o estat.o

notas: $(OBJ)
	$(CC) $(OBJ) -o notas $(LDLIBS)

main.o:   main.c   turma.h
turma.o:  turma.c  turma.h matriz.h estat.h
matriz.o: matriz.c matriz.h
estat.o:  estat.c  estat.h

.PHONY: clean
clean:
	rm -f $(OBJ) notas


In [ ]:
!make clean
!make
print('--- e agora, sem mexer em nada: ---')
!make


In [ ]:
print('--- tocando estat.c: so estat.o e a ligacao ---')
!touch estat.c && make
print('--- tocando turma.h: quem inclui o header cai junto ---')
!touch turma.h && make


**Experimento (o bug que some sozinho):** apague `turma.h` da linha de dependências de
`main.o` no `Makefile`. Agora acrescente um campo `int id;` **no início** da `struct Turma`
em `turma.h`, rode `make` e execute o programa. O `make` diz que está tudo em dia, mas
`main.o` e `turma.o` passaram a discordar sobre onde cada campo começa.

Depois rode `make clean && make` e veja o problema desaparecer — é exatamente por isso que
esse tipo de bug é tão traiçoeiro.


> Exercicio 3: acrescente ao Makefile um alvo `run` que compila e executa o programa,
> e um alvo `rebuild` que faz clean e depois compila tudo de novo.
> TODO: reescreva o Makefile abaixo com os dois alvos novos (lembre do .PHONY)


In [ ]:
%%writefile Makefile
CC      = gcc
CFLAGS  = -Wall -Wextra -g
LDLIBS  = -lm
OBJ     = main.o turma.o matriz.o estat.o

notas: $(OBJ)
	$(CC) $(OBJ) -o notas $(LDLIBS)

main.o:   main.c   turma.h
turma.o:  turma.c  turma.h matriz.h estat.h
matriz.o: matriz.c matriz.h
estat.o:  estat.c  estat.h

# TODO: alvos run e rebuild aqui

.PHONY: clean
clean:
	rm -f $(OBJ) notas


In [ ]:
!make run && make rebuild && echo 'OK: os dois alvos funcionam' \
  || echo 'Ainda nao: confira os nomes dos alvos e o .PHONY'


## 6. Biblioteca estática (`.a`)

Uma biblioteca estática é um pacote de `.o`. Na ligação, o ligador copia para dentro do
executável apenas os objetos que fazem falta.


In [ ]:
!make clean
!gcc -Wall -Wextra -c matriz.c estat.c turma.c
!ar rcs libnotas.a matriz.o estat.o turma.o
!ar t libnotas.a
!ls -l libnotas.a


In [ ]:
# agora o main pode ser compilado e ligado contra a biblioteca
!rm -f *.o
!gcc main.c -I. -L. -lnotas -lm -o notas_est
!./notas_est
!ls -l notas_est


Repare que os `.o` foram apagados antes da ligação: tudo o que o executável precisava já
estava dentro da `.a`. E depois de ligado, o executável é autossuficiente — a `.a` pode
sumir que ele continua rodando.

**Experimento:** rode `!gcc -L. -lnotas main.c -lm -o x` (biblioteca **antes** do `main.c`).
Por que falha?


## 7. Biblioteca compartilhada (`.so`)

A compartilhada não entra no executável: fica só o nome dela anotado lá dentro, e quem a
procura é o carregador do sistema, na hora de executar.


In [ ]:
!gcc -fPIC -c matriz.c estat.c turma.c
!gcc -shared -o libnotas.so matriz.o estat.o turma.o -lm
!gcc main.c -I. -L. -lnotas -o notas_din
!ls -l libnotas.so notas_din


In [ ]:
# ligou sem reclamar --- mas e na hora de rodar?
!./notas_din || echo '>>> nao encontrou a biblioteca'
print('--- dizendo onde procurar: ---')
!LD_LIBRARY_PATH=. ./notas_din
!ldd notas_din | head -5


Compare os tamanhos: o executável ligado estaticamente carrega o código dos três módulos;
o dinâmico carrega só o `main` e uma anotação. Se dez programas usarem a mesma `.so`, existe
uma cópia só — e corrigir um bug na biblioteca corrige os dez sem religar nada.

Em troca, o arquivo precisa estar lá na hora certa: é essa a falha que você acabou de ver.


### A biblioteca que você já usava sem perceber

`estat.c` chama `sqrt`. O `math.h` só traz a *declaração*; o código está em `libm`.


In [ ]:
!rm -f *.o
!gcc -Wall -c matriz.c estat.c turma.c main.c
!gcc matriz.o estat.o turma.o main.o -o sem_lm || echo '>>> undefined reference: faltou -lm'
!gcc matriz.o estat.o turma.o main.o -o com_lm -lm && echo 'OK com -lm'


Em alguns sistemas (macOS, e versões recentes da glibc) a `libm` já vem junto da `libc` e o
erro não aparece. Escreva o `-lm` assim mesmo: o mesmo código precisa compilar na máquina do
colega.


## Tarefas

### Tarefa 1 — um módulo novo

Crie `io.h`/`io.c` com `void turma_le(Turma *t)`, que lê as notas do teclado (ou de um vetor
fixo, se preferir testar sem entrada), e tire essa responsabilidade do `main.c`. Acrescente
as regras correspondentes ao `Makefile` — inclusive a dependência de header.


> Tarefa 1
> TODO: escreva io.h com a guarda e o prototipo de turma_le


In [ ]:
%%writefile io.h
/* TODO */


> TODO: escreva io.c incluindo io.h e turma.h


In [ ]:
%%writefile io.c
/* TODO */


In [ ]:
# TODO: atualize main.c para usar turma_le, e o Makefile para compilar io.o
!make && ./notas


### Tarefa 2 — biblioteca a partir do seu módulo

Empacote `matriz.o`, `estat.o`, `turma.o` e `io.o` em `libturma.a` e ligue o programa contra
ela. Apague os `.o` antes de ligar: se o executável ainda se formar, a biblioteca está
completa.


In [ ]:
# Tarefa 2
# TODO: gere libturma.a e ligue main.c contra ela, sem os .o soltos
# !ar rcs ...
# !gcc main.c -I. -L. -l... -lm -o notas_lib


### Tarefa 3 — quebrando o contrato de propósito

Acrescente um campo no **início** da `struct Turma` em `turma.h`. Recompile **só**
`turma.c` na mão (`gcc -c turma.c`), religue e execute. Explique, em duas ou três linhas,
o que apareceu na tela e por quê. Depois conserte com `make clean && make`.


In [ ]:
# Tarefa 3
# TODO: siga o roteiro acima e escreva sua explicacao no comentario abaixo
#
# Explicacao:
#


## Desafio — uma biblioteca de verdade, com tipo opaco

Reescreva o módulo da turma de modo que `turma.h` contenha apenas

```c
typedef struct Turma Turma;   /* sem os campos! */
```

Requisitos:

1. a `struct` completa fica em `turma.c`; quem usa a biblioteca chega às notas apenas por
   `turma_set(Turma *t, int i, int j, double nota)` e `turma_get(const Turma *t, int i, int j)`;
2. gere `libturma.a` **e** `libturma.so` a partir dos mesmos fontes, com dois alvos no `Makefile`;
3. escreva um cliente em **outro diretório** (`cliente/uso.c`), compilado com `-I` e `-L`,
   sem copiar nenhum `.c`;
4. explique por que, agora, é impossível o cliente depender do formato interno da `struct` —
   e o que isso significa na hora de mudar a implementação.

Dica: com o tipo opaco, `main.c` não pode mais escrever `t->notas[i][j]`. Se ele ainda
compila, algum campo escapou para o header.


In [ ]:
# Desafio
# TODO: sua solucao aqui (use %%writefile para cada arquivo e celulas !gcc/!make para testar)


## Referências

- Kernighan & Ritchie, *The C Programming Language*, 2ª ed. — cap. 4 (escopo, `static`, `extern`).
- K. N. King, *C Programming: A Modern Approach*, 2ª ed. — cap. 15 (programas com vários arquivos).
- Bryant & O'Hallaron, *Computer Systems: A Programmer's Perspective*, 3ª ed. — cap. 7 (ligação).
- Mecklenburg, *Managing Projects with GNU Make*, 3ª ed.

A lista completa está em `../referencias.bib`.
